# Дело II-06 · Экзамен для «Компаса»

**Бюро аналитических расследований, 16–17 апреля 2026 года.** Закупочная комиссия ждёт финальный ответ. Поставщик показывает почти идеальную оценку распознавания цифр, но в таблице появились `source_id`, варианты снимков и ранее не раскрытая партия C.

Ваша задача — проверить не только строки, но и единицу независимости. Связанные снимки одного источника не должны пересекать обучающую и тестовую выборки. После выбора процедуры вы один раз откроете партию C, оцените сдвиг, построите бутстреп-интервал с пересэмплированием источников и подготовите карточку модели и закупочную записку.

**Важно:** набор этого дела — документированная **учебная синтетическая стресс-проверка**. Все изменения изображений искусственны; они не описывают реальный сканер или реального поставщика.

Ориентир времени — 6–8 часов.


## Маршрут финального аудита

1. Проверить контрольную сумму и метки синтетического происхождения.
2. Воспроизвести построчное разбиение поставщика замороженной моделью II-05.
3. Найти связанные `source_id` по обе стороны границы.
4. Заменить построчное разбиение на `StratifiedGroupKFold`.
5. После выбора модели открыть партию C ровно один раз.
6. Рассчитать macro-F1, классы ошибок и бутстреп-интервал по источникам.
7. Выпустить карточку модели и решение о закупке без обвинения, которого данные не доказывают.


In [ ]:
from __future__ import annotations

import hashlib
import random
import urllib.request
import zipfile
from pathlib import Path

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
NOTEBOOK_VARIANT = "learner"
CASE_SLUG = "case-06"
ARCHIVE_NAME = "part-2-case-06.zip"
COURSE_SITE = "https://mkuziuk.github.io/python-tutorial"
IN_COLAB = False

# При локальном запуске используем файлы из каталога дела; в Colab скачиваем архив и проверяем его SHA-256.
try:
    import google.colab  # type: ignore[import-not-found]  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def find_local_case() -> Path | None:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (
            (candidate / "README.md").is_file()
            and (candidate / f"{CASE_SLUG}.ipynb").is_file()
        ):
            return candidate
        nested = candidate / "projects" / "part-2" / CASE_SLUG
        if (nested / "README.md").exists():
            return nested
    return None

def download_colab_case() -> Path:
    destination = Path("/content") / f"python-tutorial-{CASE_SLUG}"
    destination.mkdir(parents=True, exist_ok=True)
    archive_path = destination / ARCHIVE_NAME
    archive_url = f"{COURSE_SITE}/downloads/{ARCHIVE_NAME}"
    checksum_url = f"{archive_url}.sha256"

    urllib.request.urlretrieve(archive_url, archive_path)
    # Сверяем SHA-256 до распаковки, чтобы не выполнить код из повреждённого архива.
    with urllib.request.urlopen(checksum_url) as response:
        expected = response.read().decode("utf-8").split()[0].lower()
    actual = sha256_file(archive_path)
    if actual != expected:
        raise RuntimeError(f"SHA-256 архива не совпал: {actual} != {expected}")

    unpacked = destination / "unpacked"
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(unpacked)
    matches = sorted(unpacked.rglob(f"{CASE_SLUG}.ipynb"))
    if not matches:
        raise FileNotFoundError(f"В архиве нет {CASE_SLUG}.ipynb")
    return matches[0].parent

# DATA_DIR и ARTIFACTS_DIR строятся от найденного каталога дела, поэтому текущая папка не влияет на пути.
CASE_DIR = find_local_case()
if CASE_DIR is None and IN_COLAB:
    CASE_DIR = download_colab_case()
if CASE_DIR is None:
    raise FileNotFoundError(
        f"Не найден каталог {CASE_SLUG}. Запустите тетрадь из каталога дела."
    )

DATA_DIR = CASE_DIR / "data"
ARTIFACTS_DIR = CASE_DIR / "artifacts"
ARTIFACTS_DIR.mkdir(exist_ok=True)
print(f"Среда: {'Colab' if IN_COLAB else 'local'} | дело: {CASE_DIR}")
print(f"RANDOM_STATE = {RANDOM_STATE}")


In [ ]:
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    f1_score,
    recall_score,
)
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

np.random.seed(RANDOM_STATE)
pd.set_option("display.max_colwidth", 90)
sns.set_theme(style="whitegrid", context="notebook")
warnings.filterwarnings("ignore", category=FutureWarning)


## 1. Вскрываем только карточку данных

`SOURCE.md` описывает генератор. Базовые изображения взяты из встроенного Digits. Для A/B к каждому исходнику добавлен стабильный случайный «отпечаток источника» и слабый шум; четыре близких варианта распределены между обучающей и тестовой выборками поставщика. Партия C содержит другие исходники и дополнительное синтетическое горизонтальное размазывание (smear) с сильным шумом.

Это намеренно сконструированный учебный пример утечки и сдвига распределения. Поля `instructional_synthetic` и `synthetic_transform` запрещают принять его за естественные измерения.


In [ ]:
generation_report = json.loads(
    (DATA_DIR / "generation_report.json").read_text(encoding="utf-8")
)
capture_path = DATA_DIR / "digits_compass.csv.gz"
assert sha256_file(capture_path) == generation_report["sha256"], "Derivative изменён"

captures = pd.read_csv(capture_path)
pixel_columns = [f"pixel_{row}_{column}" for row in range(8) for column in range(8)]
required_metadata = {
    "sample_id", "source_id", "variant_id", "scanner_batch",
    "vendor_split", "digit", "instructional_synthetic", "synthetic_transform"
}
assert required_metadata.issubset(captures.columns)
assert captures["sample_id"].is_unique
assert captures["instructional_synthetic"].eq(True).all()
print(f"Строк: {len(captures):,}; источников: {captures['source_id'].nunique():,}")
display(pd.crosstab(captures["scanner_batch"], captures["vendor_split"], margins=True))
display(captures[["sample_id", "source_id", "variant_id", "scanner_batch", "vendor_split", "synthetic_transform"]].head())


In [ ]:
gallery_rows = captures.groupby("scanner_batch", sort=True).head(4)
fig, axes = plt.subplots(3, 4, figsize=(8, 6))
for ax, (_, row) in zip(axes.ravel(), gallery_rows.iterrows(), strict=True):
    ax.imshow(row[pixel_columns].to_numpy(dtype=float).reshape(8, 8), cmap="gray_r", vmin=0, vmax=16)
    ax.set_title(f"batch {row['scanner_batch']} · y={row['digit']}")
    ax.axis("off")
fig.suptitle("Только синтетические учебные captures")
plt.tight_layout()
plt.show()


## 2. Воспроизводим заявленную границу

Модель уже заморожена в II-05: `StandardScaler → RBF SVC(C=2, gamma='scale')`. Мы не подбираем её по этому датасету. Сначала честно воспроизводим то, что названо тестовой выборкой поставщика, а затем проверяем, независимы ли его строки.


In [ ]:
def locked_model():
    return make_pipeline(
        StandardScaler(),
        SVC(kernel="rbf", C=2, gamma="scale"),
    )

# Партии A/B образуют данные разработки; изолированная до финала партия C пока недоступна для выбора.
development = captures[captures["scanner_batch"].isin(["A", "B"])].copy()
batch_c = captures[captures["scanner_batch"].eq("C")].copy()
vendor_train_mask = development["vendor_split"].eq("train").to_numpy()
vendor_test_mask = development["vendor_split"].eq("test").to_numpy()
X_development = development[pixel_columns].to_numpy()
y_development = development["digit"].to_numpy()
source_groups = development["source_id"].to_numpy()


In [ ]:
# Сначала воспроизводим построчное разбиение поставщика, затем проверяем пересечение source_id между его частями.
vendor_model = locked_model()
vendor_model.fit(X_development[vendor_train_mask], y_development[vendor_train_mask])
vendor_predictions = vendor_model.predict(X_development[vendor_test_mask])
# TODO: посчитайте macro-F1 на тестовой выборке поставщика.
vendor_macro_f1 = float("nan")
print("Модель воспроизведена; формула macro-F1 ждёт TODO.")


### Упражнение: строки независимы или только уникальны?

`sample_id` уникален — но этого недостаточно. Сравните множества `source_id` в обучающую и тестовую выборки поставщика и посмотрите варианты нескольких пересекающихся источников. Если один оригинал породил строки с обеих сторон, такая тестовая выборка показывает, насколько модель узнаёт близкий снимок знакомого источника.


In [ ]:
# TODO: постройте множества source_id train/test и их пересечение.
vendor_train_sources: set[str] = set()
vendor_test_sources: set[str] = set()
leaked_sources: set[str] = set()
print("Временный результат: пересечение не рассчитано. Выполните TODO.")


## 3. Групповая проверка

`StratifiedGroupKFold` одновременно старается сохранить классы и удерживает все варианты `source_id` в одной части выборки. У него нет магии: мы обязаны передать правильные `groups` и проверить непересечение.

**Упражнение:** получите OOF-прогнозы для всех A/B строк. Предобработка остаётся внутри конвейера.


In [ ]:
grouped_cv = StratifiedGroupKFold(
    n_splits=3, shuffle=True, random_state=RANDOM_STATE
)
fold_audit = []
for fold_number, (train_positions, validation_positions) in enumerate(
    grouped_cv.split(X_development, y_development, groups=source_groups), start=1
):
    # Ни один источник не должен появиться одновременно в обучении и валидации.
    overlap = set(source_groups[train_positions]) & set(source_groups[validation_positions])
    fold_audit.append({
        "fold": fold_number,
        "train_rows": len(train_positions),
        "validation_rows": len(validation_positions),
        "source_overlap": len(overlap),
    })
fold_audit = pd.DataFrame(fold_audit)
display(fold_audit)


In [ ]:
# TODO: получите вневыборочные прогнозы, сохраняя группы источников целиком.
fallback = DummyClassifier(strategy="most_frequent").fit(X_development, y_development)
grouped_predictions = fallback.predict(X_development)
grouped_macro_f1 = f1_score(y_development, grouped_predictions, average="macro")
print(f"Временная базовая оценка macro-F1: {grouped_macro_f1:.3f}")


In [ ]:
grouped_recall = pd.Series(
    recall_score(y_development, grouped_predictions, labels=np.arange(10), average=None),
    index=np.arange(10),
    name="grouped_recall",
)
display(grouped_recall.to_frame().round(3))


## 4. Модель выбрана — открываем партию C

Только теперь обучаем замороженную модель на всех A/B и один раз строим прогноз для партии C. Источники C не встречались в A/B. Сдвиг распределения документирован генератором; он не обнаружен постфактум ради драматического результата.


In [ ]:
# После фиксации групповой оценки проверяем отсутствие общих source_id и оцениваем модель на партии C.
assert set(development["source_id"]).isdisjoint(batch_c["source_id"])
final_model = locked_model().fit(X_development, y_development)
X_batch_c = batch_c[pixel_columns].to_numpy()
y_batch_c = batch_c["digit"].to_numpy()
batch_c_predictions = final_model.predict(X_batch_c)
batch_c_macro_f1 = f1_score(y_batch_c, batch_c_predictions, average="macro")

metric_comparison = pd.DataFrame({
    "evaluation": ["построчное разбиение поставщика", "source-grouped OOF", "sealed scanner C"],
    "macro_f1": [vendor_macro_f1, grouped_macro_f1, batch_c_macro_f1],
}).set_index("evaluation")
display(metric_comparison.round(3))


In [ ]:
c_report = pd.DataFrame(
    classification_report(
        y_batch_c, batch_c_predictions, labels=np.arange(10),
        output_dict=True, zero_division=0,
    )
).T
display(c_report.loc[[str(i) for i in range(10)], ["precision", "recall", "f1-score", "support"]].round(3))

fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay.from_predictions(
    y_batch_c, batch_c_predictions, labels=np.arange(10), cmap="Blues", ax=ax,
    colorbar=False,
)
ax.set_title("Scanner C: confusion matrix после вскрытия")
ax.grid(False)
plt.show()


### Упражнение: бутстреп-интервал с пересэмплированием источников

Ресемплируйте `source_id`, а не отдельные строки. Для каждой бутстреп-выборки включите все строки выбранного источника и пересчитайте macro-F1 с фиксированными метками классов 0–9. Здесь у C один снимок на источник, но функция остаётся корректной и при нескольких вариантах.

Интервал отражает вариативность этой конечной выборки источников; он не исправляет систематический сдвиг распределения и не является гарантией будущего качества.


In [ ]:
# TODO: реализуйте ресемплинг source_id и пересчёт macro-F1.
def source_bootstrap_interval(
    y_true, y_pred, groups, *, repeats=1_000, confidence=0.95, random_state=42
):
    return (float("nan"), float("nan"))

batch_c_interval = source_bootstrap_interval(
    y_batch_c, batch_c_predictions, batch_c["source_id"].to_numpy()
)
print("Bootstrap CI ждёт TODO.")


In [ ]:
c_errors = np.flatnonzero(batch_c_predictions != y_batch_c)
shown = c_errors[:12]
fig, axes = plt.subplots(3, 4, figsize=(8, 6))
for ax in axes.ravel():
    ax.axis("off")
for ax, position in zip(axes.ravel(), shown, strict=False):
    ax.imshow(X_batch_c[position].reshape(8, 8), cmap="gray_r", vmin=0, vmax=16)
    ax.set_title(f"истина {y_batch_c[position]} → {batch_c_predictions[position]}")
    ax.axis("off")
fig.suptitle("Ошибки на синтетическом scanner C")
plt.tight_layout()
plt.show()


## 5. Карточка модели и закупочная записка

Карточка модели должна назвать назначение, данные, модель, три оценки, ограничения и запрещённые применения. Закупочная записка отделяет методологический дефект от персонального обвинения: подгонка процедуры делает заявленный результат непригодным для решения, но сама по себе не устанавливает умысел или мошенничество.


In [ ]:
# TODO: заполните карточку модели и записку своими числами и ограничениями.
model_card = {
    "model": "StandardScaler → RBF SVC(C=2, gamma='scale')",
    "intended_use": "TODO",
    "data": "Учебная синтетическая производная Digits.",
    "evaluation": {
        "vendor_row_split_macro_f1": vendor_macro_f1,
        "source_grouped_oof_macro_f1": grouped_macro_f1,
        "scanner_c_macro_f1": batch_c_macro_f1,
        "scanner_c_source_bootstrap_95pct": list(batch_c_interval),
    },
    "limitations": ["TODO"],
    "prohibited_uses": ["TODO"],
}
procurement = {
    "established_fact": "TODO",
    "supported_interpretation": "TODO",
    "not_proven": "TODO",
    "limitations": "TODO",
    "recommended_action": "TODO",
}
(ARTIFACTS_DIR / "model_card.md").write_text("# Model card\n\nTODO\n", encoding="utf-8")
(ARTIFACTS_DIR / "procurement_memo.md").write_text("# Закупочная записка\n\nTODO\n", encoding="utf-8")
print("Черновики записаны; заполните TODO.")


In [ ]:
if NOTEBOOK_VARIANT == "solution":
    vendor_gap = vendor_macro_f1 - grouped_macro_f1
    shift_gap = grouped_macro_f1 - batch_c_macro_f1
    assert len(leaked_sources) == development["source_id"].nunique()
    assert fold_audit["source_overlap"].eq(0).all()
    assert vendor_gap >= 0.05
    assert shift_gap >= 0.05
    assert 0.87 <= grouped_macro_f1 <= 0.91
    assert 0.61 <= batch_c_macro_f1 <= 0.69
    assert 0 <= batch_c_interval[0] <= batch_c_macro_f1 <= batch_c_interval[1] <= 1
    assert set(procurement) == {
        "established_fact", "supported_interpretation", "not_proven",
        "limitations", "recommended_action"
    }
    print(
        f"Проверки II-06 пройдены: vendor→group gap={vendor_gap:.3f}; "
        f"group→C gap={shift_gap:.3f}."
    )
else:
    print("Учебный вариант: строгие проверки включатся после сверки с решением.")


## Финал протокола «Компас»

Выполните **Restart Kernel → Run All** и сверьте `check_result.md`. Финальный вывод бюро: заявленная контрольная оценка непригодна для решения о внедрении, закупка приостановлена до независимой проверки на данных целевой задачи. Данные не доказывают личное мошенничество — этот предел вывода так же важен, как найденные разрывы метрик.

**Типичная ошибка:** открыть партию C, увидеть спад и начать менять модель. Тогда C перестаёт быть изолированной проверкой. Следующая оценка потребует новой независимой партии.

**Расширение:** до открытия новой партии сформулируйте приёмочный порог и минимальные размеры каждого класса, затем симулируйте решение комиссии без дополнительного подбора.
